In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_15_7_0,0.999963,0.317567,0.999912,0.999787,0.999845,0.000022,0.405122,3.581356e-05,9.975416e-05,6.778386e-05,0.001271,0.004657,1.000067,0.004856,95.477242,140.575647,"Hidden Size=[4, 4], regularizer=0.3, learning_..."
1,model_15_8_11,0.999963,0.317567,0.999995,1.000000,0.999996,0.000022,0.405122,3.263664e-06,2.341201e-13,1.631832e-06,0.001271,0.004658,1.000067,0.004856,95.477048,140.575453,"Hidden Size=[4, 4], regularizer=0.3, learning_..."
2,model_15_8_2,0.999963,0.317567,0.999995,1.000000,0.999996,0.000022,0.405122,3.263664e-06,2.341201e-13,1.631832e-06,0.001271,0.004658,1.000067,0.004856,95.477048,140.575453,"Hidden Size=[4, 4], regularizer=0.3, learning_..."
3,model_15_7_16,0.999963,0.317567,0.999912,0.999787,0.999845,0.000022,0.405122,3.581950e-05,9.976137e-05,6.779020e-05,0.001271,0.004658,1.000067,0.004856,95.477048,140.575453,"Hidden Size=[4, 4], regularizer=0.3, learning_..."
4,model_15_7_15,0.999963,0.317567,0.999912,0.999787,0.999845,0.000022,0.405122,3.581950e-05,9.976137e-05,6.779020e-05,0.001271,0.004658,1.000067,0.004856,95.477048,140.575453,"Hidden Size=[4, 4], regularizer=0.3, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,model_14_5_13,0.997508,0.594808,1.000000,1.000000,1.000000,0.001479,0.240539,1.029482e-13,3.968398e-14,7.398398e-14,0.015768,0.038461,1.004600,0.040098,87.032466,132.130872,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1544,model_14_5_14,0.997508,0.594808,1.000000,1.000000,1.000000,0.001479,0.240539,1.029482e-13,3.968398e-14,7.398398e-14,0.015768,0.038461,1.004600,0.040098,87.032466,132.130872,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1545,model_14_5_15,0.997508,0.594808,1.000000,1.000000,1.000000,0.001479,0.240539,1.029482e-13,3.968398e-14,7.398398e-14,0.015768,0.038461,1.004600,0.040098,87.032466,132.130872,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
1546,model_14_5_16,0.997508,0.594808,1.000000,1.000000,1.000000,0.001479,0.240539,1.029482e-13,3.968398e-14,7.398398e-14,0.015768,0.038461,1.004600,0.040098,87.032466,132.130872,"Hidden Size=[4, 4], regularizer=0.5, learning_..."
